# Unified Pole + Crossarm + Wire Detection

**Single notebook — no intermediate files, no manual labels required.**

Everything is derived from scratch from `laser_left.las`.

### Pipeline stages
| Stage | What happens | Source logic |
|---|---|---|
| 0 | Load `laser_left.las`, apply global coordinate shift | `poles.ipynb` |
| 1 | PDAL SMRF ground classification → Height Above Ground (HAG) | `poles.ipynb` |
| 2 | Eigenvalue pre-screen → linear/planar candidates | `poles.ipynb` |
| 3 | DBSCAN clustering → candidate pole structures | `poles.ipynb` |
| 4 | Height/width filter + 3-D expansion with crossarm recovery | `poles.ipynb` |
| 5 | Tree rejection — HAG + spread guard (no manual labels) | `maybe_no_wire_2_.ipynb` |
| 6 | Component labelling (SHAFT / CROSSARM / TRANSFORMER) | `poles.ipynb` |
| 7 | KMeans lane split (left / right side of road) | `maybe_no_wire_2_.ipynb` |
| 8 | Robust pole-top estimation from raw cloud | `poles.ipynb` |
| 9 | Wire detection: 3-D tube fetch → catenary fit → eigenvalue linearity/planarity filter | `non_pro_v1.ipynb` |
|10 | Write `poles_wires.las` + `poles_wires.qml` + `span_report.csv` | all three |

**Input:** `laser_left.las`  
**Outputs:** `poles_wires.las`, `poles_wires.qml`, `span_report.csv`

### Class legend
| LAS class | Name | Colour |
|---|---|---|
| 65 | SHAFT | red |
| 66 | CROSSARM | amber |
| 72 | TRANSFORMER | purple |
| 70 | SAME_LANE_WIRE | blue |
| 71 | CROSS_ROAD_WIRE | red |

## Stage 0 — Load & global shift

All processing is done in **local (shifted) coordinates** to avoid floating-point precision loss at UTM-scale numbers. The shift is added back only when writing the final LAS.

In [ ]:
import numpy as np
import pandas as pd
import laspy
from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN, KMeans

# ── paths ──────────────────────────────────────────────────────────────────────
RAW_LAS    = 'laser_left.las'
OUT_LAS    = 'poles_wires.las'
OUT_QML    = 'poles_wires.qml'
OUT_SPANS  = 'span_report.csv'

# ── global shift (from poles.ipynb — keeps arithmetic stable) ──────────────────
LOCAL_SHIFT = np.array([356534.91, 4661602.80, 215.93])

# ── load ───────────────────────────────────────────────────────────────────────
print('Loading laser_left.las ...')
las    = laspy.read(RAW_LAS)
points = np.vstack((las.x, las.y, las.z)).T          # global UTM coords
local  = points - LOCAL_SHIFT                         # shifted local coords
print(f'  {len(local):,} points loaded')
print(f'  Local bbox X [{local[:,0].min():.1f}, {local[:,0].max():.1f}]')
print(f'            Y [{local[:,1].min():.1f}, {local[:,1].max():.1f}]')
print(f'            Z [{local[:,2].min():.1f}, {local[:,2].max():.1f}]')

# ── spatial indices on local coords ───────────────────────────────────────────
# Decimated 2-D tree: fast candidate screening
decimated   = local[::5]
tree_dec_2d = cKDTree(decimated[:, :2])

# Full 2-D and 3-D trees on ALL points (needed for wire fetching)
tree_full_2d = cKDTree(local[:, :2])
tree_full_3d = cKDTree(local)           # 3-D — critical for wire Z capture

print('Spatial indices built.')

## Stage 1 — PDAL SMRF: Height Above Ground

Ground points are classified with SMRF; HAG is computed for every point.
Poles and wires have HAG > 4 m; ground/undergrowth has HAG ≈ 0.

In [ ]:
import pdal
import json

pdal_pipeline = [
    {"type": "readers.las",    "filename": RAW_LAS},
    {"type": "filters.smrf"},
    {"type": "filters.hag_nn"},
    {"type": "writers.las",    "filename": "_temp_hag.las", "extra_dims": "all"}
]

print('Running PDAL SMRF + HAG ...')
p = pdal.Pipeline(json.dumps(pdal_pipeline))
p.execute()

las_hag = laspy.read('_temp_hag.las')
hag     = np.array(las_hag.HeightAboveGround)   # same order as `local`
print(f'  HAG range: {hag.min():.1f} – {hag.max():.1f} m')
print(f'  Points with HAG > 4 m: {(hag > 4).sum():,}  (pole/wire candidates)')
print('SMRF complete.')

## Stage 2 — Eigenvalue pre-screen on decimated cloud

Compute per-point linearity and planarity on the decimated cloud.
Vertical linear structures (poles) have **linearity > 0.75, planarity < 0.25**.

In [ ]:
EIGEN_RADIUS   = 1.2   # neighbourhood radius (m)
MIN_NEIGHBOURS = 8     # minimum points to compute geometry
MIN_LINEARITY  = 0.75
MAX_PLANARITY  = 0.25

def extract_signatures(pts_3d):
    """Return (linearity, planarity, sphericity, z_dir) for a local neighbourhood."""
    if len(pts_3d) < 5:
        return 0, 0, 0, 0
    pts_c = pts_3d - pts_3d.mean(axis=0)
    cov   = np.cov(pts_c.T) + np.eye(3) * 1e-6
    evals, evecs = np.linalg.eigh(cov)
    idx  = evals.argsort()[::-1]
    L1, L2, L3 = evals[idx]
    linearity  = (L1 - L2) / L1 if L1 > 0 else 0
    planarity  = (L2 - L3) / L1 if L1 > 0 else 0
    sphericity = L3 / L1         if L1 > 0 else 0
    z_dir      = abs(evecs[2, idx[0]])
    return linearity, planarity, sphericity, z_dir

print('Running eigenvalue pre-screen on decimated cloud ...')
pole_cand_idx = []

for i in range(0, len(decimated), 10):   # stride-10 for speed
    nbr_idx  = tree_dec_2d.query_ball_point(decimated[i, :2], r=EIGEN_RADIUS)
    if len(nbr_idx) <= MIN_NEIGHBOURS:
        continue
    nbr_pts  = decimated[nbr_idx]
    lin, pla, _, _ = extract_signatures(nbr_pts)
    if lin > MIN_LINEARITY and pla < MAX_PLANARITY:
        pole_cand_idx.append(i)

candidate_pts = decimated[pole_cand_idx]
print(f'  {len(candidate_pts):,} eigen-candidate points')

## Stage 3 — DBSCAN clustering → pole structures

In [ ]:
DBSCAN_EPS      = 2.0
DBSCAN_MIN_SAMP = 10
MIN_HEIGHT      = 4.0   # m  — minimum cluster vertical extent
MAX_WIDTH       = 4.0   # m  — maximum horizontal spread

print('DBSCAN clustering ...')
db     = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMP).fit(candidate_pts)
labels = db.labels_

initial_clusters = []
for lbl in set(labels):
    if lbl == -1:
        continue
    cl     = candidate_pts[labels == lbl]
    extent = cl.max(axis=0) - cl.min(axis=0)
    h, w   = extent[2], max(extent[0], extent[1])
    if h > MIN_HEIGHT and w < MAX_WIDTH:
        initial_clusters.append(cl)

print(f'  {len(initial_clusters)} clusters pass height/width filter')

## Stage 4 — 3-D expansion: recover full structure + crossarms

For each cluster we expand into the **full-density** raw cloud to recover the crossarm tips and any points missed by decimation.

In [ ]:
EXPAND_XY_RADIUS = 2.5   # m — wide enough for crossarms
EXPAND_Z_BUFFER  = 6.0   # m — look this far above the cluster top
TIGHT_RADIUS     = 1.2   # m — fallback tight cylinder if spread is too large
MAX_SPREAD       = 1.5   # m — std dev threshold for tree rejection at expansion

expanded_clusters = []

for cl in initial_clusters:
    cxy   = cl[:, :2].mean(axis=0)
    max_z = cl[:, 2].max()
    min_z = cl[:, 2].min()

    # Distance filter on the full cloud (vectorised, no tree call)
    dist_sq = ((local[:, 0] - cxy[0])**2 +
               (local[:, 1] - cxy[1])**2)
    mask = (
        (dist_sq < EXPAND_XY_RADIUS**2) &
        (local[:, 2] >= min_z) &
        (local[:, 2] <= max_z + EXPAND_Z_BUFFER)
    )
    full = local[mask]
    if len(full) == 0:
        continue

    spread = np.std(full[:, :2])
    if spread < MAX_SPREAD:
        expanded_clusters.append(full)
    else:
        # Too wide → keep only the tight cylinder to cut away tree branches
        tight = np.linalg.norm(full[:, :2] - cxy, axis=1) < TIGHT_RADIUS
        expanded_clusters.append(full[tight])

print(f'Expansion complete: {len(expanded_clusters)} full structures')

## Stage 5 — Tree rejection using HAG + spread

A genuine pole structure has:
- Median HAG of the cluster > `MIN_MEDIAN_HAG` (trees have a broad HAG distribution)
- A high proportion of points with HAG > 4 m (pole body is continuously elevated)
- Horizontal XY spread is narrow (poles are thin cylinders)

This removes tree blobs **without needing any manual labels**.

In [ ]:
MIN_MEDIAN_HAG    = 3.0   # m  — median height above ground
MIN_HIGH_FRAC     = 0.30  # fraction of cluster points with HAG > 4 m
MAX_XY_STD        = 1.0   # m  — horizontal spread std (poles are narrow)

# Build a KDTree on the HAG-indexed points (full cloud, same order as `local`)
# We map each expanded cluster point back to its HAG value via nearest-neighbour
# (The full cloud points ARE in `local`, so we can index directly by original index)
# Use the tree to find the global indices of each cluster's points
full_tree_3d_local = cKDTree(local)   # 3-D tree in local coords (reuse if already built)

clean_clusters = []
rejected = 0

for cl in expanded_clusters:
    # Map each cluster point → its index in `local` → its HAG value
    _, idx = full_tree_3d_local.query(cl, k=1)
    cl_hag = hag[idx]

    median_hag = float(np.median(cl_hag))
    high_frac  = float((cl_hag > 4.0).sum()) / len(cl_hag)
    xy_std     = float(np.std(cl[:, :2]))

    if median_hag < MIN_MEDIAN_HAG or high_frac < MIN_HIGH_FRAC or xy_std > MAX_XY_STD:
        rejected += 1
        continue

    clean_clusters.append(cl)

print(f'Tree rejection: kept {len(clean_clusters)} poles, rejected {rejected} tree/noise clusters')

## Stage 6 — Component labelling (SHAFT / CROSSARM / TRANSFORMER)

Simple geometric rules applied per cluster:
- **CROSSARM** — near the top (`h_from_top < 1.2 m`) AND offset from pole axis (`d_centre > 0.4 m`)
- **TRANSFORMER** — mid-height (`2 – 6 m from top`) AND offset from axis (`d_centre > 0.5 m`)
- Everything else → **SHAFT**

Also computes the **pole centre table** (X, Y, Z, height) and **pole tops** used by wire detection.

In [ ]:
CLASS_SHAFT       = 1
CLASS_CROSSARM    = 2
CLASS_TRANSFORMER = 8

CLASSES = {1:'SHAFT', 2:'CROSSARM', 3:'PRIMARY', 4:'SECONDARY',
           5:'NEUTRAL', 6:'GUY', 7:'ANCHOR', 8:'TRANSFORMER'}

pole_list   = []
pole_tops   = {}   # pid -> (global_X, global_Y, global_Z_top)
clean_pts_list  = []
clean_cls_list  = []
clean_pid_list  = []

for pid, cl in enumerate(clean_clusters):
    cxy    = cl[:, :2].mean(axis=0)
    top_z  = float(cl[:, 2].max())
    bot_z  = float(cl[:, 2].min())
    height = top_z - bot_z

    # Global coords of pole centre
    gx, gy, gz = float(cxy[0] + LOCAL_SHIFT[0]), \
                 float(cxy[1] + LOCAL_SHIFT[1]), \
                 float(cl[:, 2].mean() + LOCAL_SHIFT[2])
    pole_tops[pid] = (gx, gy, top_z + LOCAL_SHIFT[2])   # global top Z

    pole_list.append({'Pole_ID': pid, 'X': gx, 'Y': gy, 'Z': gz, 'Height': round(height, 2)})

    # Per-point class labels (work in local coords)
    d_centre   = np.linalg.norm(cl[:, :2] - cxy, axis=1)
    h_from_top = top_z - cl[:, 2]

    labels_cl  = np.full(len(cl), CLASS_SHAFT, dtype=np.uint8)
    crossarm   = (h_from_top < 1.2)  & (d_centre > 0.4)
    transformer= (h_from_top > 2.0)  & (h_from_top < 6.0) & (d_centre > 0.5)
    labels_cl[crossarm]    = CLASS_CROSSARM
    labels_cl[transformer] = CLASS_TRANSFORMER

    # Convert back to global coords for LAS output
    clean_pts_list.append(cl + LOCAL_SHIFT)
    clean_cls_list.append(labels_cl)
    clean_pid_list.append(np.full(len(cl), pid, dtype=np.int32))

poles_df  = pd.DataFrame(pole_list)
clean_pts = np.vstack(clean_pts_list)
clean_cls = np.concatenate(clean_cls_list)
clean_pid = np.concatenate(clean_pid_list)

print(f'{len(poles_df)} poles found')
vals, cnts = np.unique(clean_cls, return_counts=True)
for v, c in zip(vals, cnts):
    print(f"  {CLASSES.get(int(v),'?'):<12}  {c:>7,}")

## Stage 7 — KMeans lane split (left / right side of road)

Split poles into two lanes by clustering on the X coordinate.
West (lower X) = lane 0, East (higher X) = lane 1.
Same-lane pairs → `SAME_LANE_WIRE`; cross-lane pairs → `CROSS_ROAD_WIRE`.

In [ ]:
km     = KMeans(n_clusters=2, random_state=0, n_init=10).fit(poles_df[['X']].values)
llabels = km.labels_
if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0]:
    llabels = 1 - llabels
poles_df['Lane'] = llabels

print(f"Lane split at X ≈ {km.cluster_centers_.mean():.1f}")
print(f"  West lane (0): {(llabels==0).sum()} poles")
print(f"  East lane (1): {(llabels==1).sum()} poles")

pole_lane = {int(r['Pole_ID']): int(r['Lane']) for _, r in poles_df.iterrows()}

## Stage 8 — Robust pole-top Z estimation

Recompute each pole's top Z using the **98th percentile** of raw points within 1.2 m of the pole centre.
This is more accurate than the cluster max (avoids outliers) and is the reference height for wire detection.

In [ ]:
POLE_TOP_RADIUS = 1.2   # m XY radius to sample raw points for top estimate
POLE_TOP_PCT    = 98    # percentile

# tree_full_2d is on LOCAL coords — pole centres from poles_df are in GLOBAL coords
# So query using global points tree
global_tree_2d = cKDTree(points[:, :2])

for _, row in poles_df.iterrows():
    pid  = int(row['Pole_ID'])
    px, py = float(row['X']), float(row['Y'])
    idxs = global_tree_2d.query_ball_point([px, py], r=POLE_TOP_RADIUS)
    if len(idxs) >= 10:
        top_z = float(np.percentile(points[idxs, 2], POLE_TOP_PCT))
    else:
        top_z = pole_tops[pid][2]   # fallback to cluster top
    pole_tops[pid] = (px, py, top_z)

top_zs = [t[2] for t in pole_tops.values()]
print(f'Pole top Z (global): min={min(top_zs):.1f}, median={np.median(top_zs):.1f}, max={max(top_zs):.1f} m')
print(f'Pole top heights above shift: min={min(top_zs)-LOCAL_SHIFT[2]:.1f}, max={max(top_zs)-LOCAL_SHIFT[2]:.1f} m')

## Stage 9 — Wire detection

### Method
For every pole pair within the span range:

1. **3-D tube fetch** — query `raw_tree_3d` (built on LOCAL coords) from the 3-D midpoint of the span. This correctly captures wire sag at any height, unlike a 2-D XY query.
2. **Parabola (catenary) fit** — fit a parabola along the span direction; reject points whose residual exceeds `CAT_RESIDUAL`.
3. **Across-span spread gate** — std of the across-axis coordinate must be < `MAX_ACROSS_STD` (eliminates tree blobs).
4. **Eigenvalue linearity + planarity** — compute λ1 ≥ λ2 ≥ λ3 of the local covariance for each surviving point. Wire points have `linearity = (λ1−λ2)/λ1 ≥ 0.65` and `planarity = (λ2−λ3)/λ1 ≤ 0.25`.
5. **Span coverage check** — wire points must cover ≥ 30% of the inter-pole distance (ensures continuity).
6. **Parabola re-expand** — refit on clean subset and pull back any wire points at the tube edge that passed the geometry but were on the boundary.

Passes A (strict) → B (loose) → C (linear fallback). First pass that yields points wins.

**Note:** All wire queries operate in **LOCAL coordinates** using `tree_full_3d`.

In [ ]:
# ── Wire detection tuning ─────────────────────────────────────────────────────
MIN_SPAN        = 5.0    # m  minimum inter-pole distance to consider
MAX_SPAN        = 90.0   # m  maximum inter-pole distance

# Tube
TUBE_RADIUS     = 1.5    # m  half-width of initial 3-D search tube
Z_BELOW         = 3.5    # m  search below the lower pole top
Z_ABOVE         = 1.5    # m  search above the higher pole top

# Catenary
CAT_RESIDUAL_A  = 0.6    # m  strict pass
CAT_RESIDUAL_B  = 1.0    # m  loose pass
MIN_SPAN_COV    = 0.30   # fraction of span that must be covered

# Eigenvalue (linearity / planarity)
WIRE_MIN_LIN    = 0.65
WIRE_MAX_PLA    = 0.25
EIG_K           = 12     # k-nearest neighbours for local covariance

# Across-span spread
MAX_ACROSS_STD  = 0.35
# ─────────────────────────────────────────────────────────────────────────────


def local_eigenfeatures(pts_local, k=EIG_K):
    """Per-point linearity and planarity from k nearest neighbours."""
    n   = len(pts_local)
    lin = np.zeros(n, dtype=np.float32)
    pla = np.ones(n,  dtype=np.float32)
    if n < 4:
        return lin, pla
    lt  = cKDTree(pts_local)
    kk  = min(k + 1, n)
    _, nbr_idx = lt.query(pts_local, k=kk)
    for i in range(n):
        nbrs = pts_local[nbr_idx[i][1:]]
        if len(nbrs) < 3:
            continue
        cov  = np.cov(nbrs.T)
        evals = np.linalg.eigvalsh(cov)
        l1, l2, l3 = sorted(evals, reverse=True)   # l1 ≥ l2 ≥ l3
        if l1 < 1e-10:
            continue
        lin[i] = (l1 - l2) / l1
        pla[i] = (l2 - l3) / l1
    return lin, pla


def project_span(pts, A_local, u):
    """Project pts (local 3-D) onto the span axis and perpendicular axis."""
    rel    = pts[:, :2] - A_local
    along  = rel @ u
    nv     = np.array([-u[1], u[0]])
    across = rel @ nv
    return along, across


def fit_parabola(s, z):
    M = np.stack([np.ones_like(s), s, s * s], axis=1)
    return np.linalg.lstsq(M, z, rcond=None)[0]


def fetch_tube(A_local, B_local, aZ_local, bZ_local, span):
    """
    Fetch candidate wire points from the full 3-D tree in LOCAL coords.
    Returns (global_indices_into_local, pts, along, across) or None.
    """
    mid2d  = (A_local + B_local) / 2
    midZ   = (aZ_local + bZ_local) / 2
    mid3d  = np.array([mid2d[0], mid2d[1], midZ])
    u      = (B_local - A_local) / span

    r      = span / 2 + TUBE_RADIUS + 2.0
    cand_g = np.array(tree_full_3d.query_ball_point(mid3d, r=r), dtype=np.int64)
    if len(cand_g) == 0:
        return None

    cpts         = local[cand_g]
    along, across = project_span(cpts, A_local, u)

    low, high = min(aZ_local, bZ_local), max(aZ_local, bZ_local)
    mask = (
        (along  >  0.3) & (along < span - 0.3) &
        (np.abs(across) < TUBE_RADIUS) &
        (cpts[:, 2] > low  - Z_BELOW) &
        (cpts[:, 2] < high + Z_ABOVE)
    )
    if mask.sum() < 5:
        return None
    return cand_g[mask], cpts[mask], along[mask], across[mask]


def wire_pipeline(tube_result, span, cat_residual):
    """
    Apply catenary → spread → eigenvalue → coverage → re-expand.
    Returns global index array of wire points (into `local`), or empty.
    """
    cand_g, cpts, along, across = tube_result

    # 1. Catenary residual
    try:
        coef   = fit_parabola(along, cpts[:, 2])
        z_pred = coef[0] + coef[1]*along + coef[2]*along*along
        keep1  = np.abs(cpts[:, 2] - z_pred) < cat_residual
    except Exception:
        return np.array([], dtype=np.int64)
    if keep1.sum() < 5:
        return np.array([], dtype=np.int64)

    # 2. Across-span spread
    if np.std(across[keep1]) > MAX_ACROSS_STD:
        return np.array([], dtype=np.int64)

    # 3. Eigenvalue linearity + planarity
    lin, pla = local_eigenfeatures(cpts[keep1])
    keep2    = (lin >= WIRE_MIN_LIN) & (pla <= WIRE_MAX_PLA)
    if keep2.sum() < 4:
        return np.array([], dtype=np.int64)

    # 4. Span coverage
    al_ok = along[keep1][keep2]
    if (al_ok.max() - al_ok.min()) / span < MIN_SPAN_COV:
        return np.array([], dtype=np.int64)

    # 5. Re-expand using cleaner parabola
    try:
        coef2   = fit_parabola(al_ok, cpts[keep1][keep2][:, 2])
        z_pred2 = coef2[0] + coef2[1]*along + coef2[2]*along*along
        keep3   = (np.abs(cpts[:, 2] - z_pred2) < cat_residual) & \
                  (np.abs(across) < MAX_ACROSS_STD * 2.5)
    except Exception:
        keep3 = keep1.copy()
        keep3[keep1] &= keep2

    out = cand_g[keep3]
    return out if len(out) >= 4 else np.array([], dtype=np.int64)


def linear_fallback(A_local, B_local, aZ_local, bZ_local, span, u):
    """
    Pass C — simple linear tube, no catenary. Last resort for near-horizontal spans.
    """
    mid2d  = (A_local + B_local) / 2
    midZ   = (aZ_local + bZ_local) / 2
    mid3d  = np.array([mid2d[0], mid2d[1], midZ])
    r      = span / 2 + TUBE_RADIUS + 2.0
    cand_g = np.array(tree_full_3d.query_ball_point(mid3d, r=r), dtype=np.int64)
    if len(cand_g) == 0:
        return np.array([], dtype=np.int64)
    cpts         = local[cand_g]
    along, across = project_span(cpts, A_local, u)
    line_z = aZ_local + (bZ_local - aZ_local) * (along / span)
    mask = (
        (along  > 0.3) & (along < span - 0.3) &
        (np.abs(across) < TUBE_RADIUS) &
        (np.abs(cpts[:, 2] - line_z) < 1.5) &
        (cpts[:, 2] > min(aZ_local, bZ_local) - 3.5)
    )
    if mask.sum() < 5:
        return np.array([], dtype=np.int64)
    if np.std(across[mask]) > MAX_ACROSS_STD:
        return np.array([], dtype=np.int64)
    al = along[mask]
    if (al.max() - al.min()) / span < MIN_SPAN_COV:
        return np.array([], dtype=np.int64)
    return cand_g[mask]


print('Wire detection helpers ready.')

In [ ]:
# ── Run detection across all pole pairs ──────────────────────────────────────
pole_ids = poles_df['Pole_ID'].values.astype(int)
pole_xys = poles_df[['X', 'Y']].values    # GLOBAL XY

# Build pairs
pairs = []
for i in range(len(pole_xys)):
    for j in range(i + 1, len(pole_xys)):
        d = float(np.linalg.norm(pole_xys[i] - pole_xys[j]))
        if MIN_SPAN < d < MAX_SPAN:
            pairs.append((i, j, d))
print(f'Candidate pairs in [{MIN_SPAN}, {MAX_SPAN}] m: {len(pairs)}')

wire_assignments = {}   # index-into-`local` → wire LAS class (70 or 71)
span_rows        = []

for (i, j, d) in pairs:
    pid_a, pid_b = int(pole_ids[i]), int(pole_ids[j])

    # Global XY pole positions
    Ag, Bg = pole_xys[i], pole_xys[j]

    # Convert to LOCAL for tree queries
    A_loc = Ag - LOCAL_SHIFT[:2]
    B_loc = Bg - LOCAL_SHIFT[:2]

    # Global top Z → local top Z
    aZ_loc = pole_tops[pid_a][2] - LOCAL_SHIFT[2]
    bZ_loc = pole_tops[pid_b][2] - LOCAL_SHIFT[2]

    vec = B_loc - A_loc
    u   = vec / d

    cross = pole_lane[pid_a] != pole_lane[pid_b]
    cls   = 71 if cross else 70
    kind  = 'CROSS_ROAD' if cross else 'SAME_LANE'

    # Pass A — strict catenary
    tube = fetch_tube(A_loc, B_loc, aZ_loc, bZ_loc, d)
    survivors = np.array([], dtype=np.int64)
    used_pass = None

    if tube is not None:
        survivors = wire_pipeline(tube, d, CAT_RESIDUAL_A)
        if len(survivors): used_pass = 'A'

    # Pass B — loose catenary
    if not len(survivors) and tube is not None:
        survivors = wire_pipeline(tube, d, CAT_RESIDUAL_B)
        if len(survivors): used_pass = 'B'

    # Pass C — linear fallback
    if not len(survivors):
        survivors = linear_fallback(A_loc, B_loc, aZ_loc, bZ_loc, d, u)
        if len(survivors): used_pass = 'C'

    if not len(survivors):
        continue

    for g in survivors:
        g = int(g)
        if g not in wire_assignments:
            wire_assignments[g] = cls

    span_rows.append({'pole_a': pid_a, 'pole_b': pid_b,
                      'span_m': round(d, 1), 'kind': kind,
                      'pass': used_pass, 'points': len(survivors)})
    print(f'  {kind:12s}  {pid_a:3d}↔{pid_b:3d}  {d:5.1f} m  pass={used_pass}  → {len(survivors):,} pts')

spans_df = pd.DataFrame(span_rows).sort_values(['kind', 'pole_a', 'pole_b'])
spans_df.to_csv(OUT_SPANS, index=False)

print(f'\nSpans detected: {len(spans_df)}')
if len(spans_df):
    for k in ['SAME_LANE', 'CROSS_ROAD']:
        n = (spans_df['kind'] == k).sum()
        if n: print(f"  {k}: {n}")
    for p in ['A','B','C']:
        n = (spans_df['pass'] == p).sum()
        if n: print(f"  Pass {p}: {n} spans")
print(f'Total unique wire points: {len(wire_assignments):,}')
print(f'{OUT_SPANS} written')

## Stage 10 — Write output LAS + QML

### LAS class map
| Class | Name | Colour |
|---|---|---|
| 65 | SHAFT | red |
| 66 | CROSSARM | amber |
| 72 | TRANSFORMER | purple |
| 70 | SAME_LANE_WIRE | blue |
| 71 | CROSS_ROAD_WIRE | crimson |

In [ ]:
LAS_OFFSET = 64   # component classes 1-8 → LAS classes 65-72

# Wire points are in LOCAL coords → convert to GLOBAL for LAS
if wire_assignments:
    wg       = np.array(list(wire_assignments.keys()), dtype=np.int64)
    wire_pts = local[wg] + LOCAL_SHIFT          # back to global
    wire_cls = np.array([wire_assignments[int(g)] for g in wg], dtype=np.uint8)
else:
    wire_pts = np.zeros((0, 3))
    wire_cls = np.zeros(0, dtype=np.uint8)

# Pole component points are already in global coords
final_pts = np.vstack([clean_pts, wire_pts])
final_cls = np.concatenate([
    clean_cls + LAS_OFFSET,    # shaft=65, crossarm=66, transformer=72
    wire_cls                   # same_lane=70, cross_road=71
])
final_pid = np.clip(np.concatenate([
    clean_pid,
    np.zeros(len(wire_pts), dtype=np.int32)
]) + 1, 0, 255).astype(np.uint8)

print(f'Writing {len(final_pts):,} total points')
for cls_val, name in [(65,'SHAFT'),(66,'CROSSARM'),(72,'TRANSFORMER'),(70,'SAME_LANE_WIRE'),(71,'CROSS_ROAD_WIRE')]:
    n = (final_cls == cls_val).sum()
    if n: print(f'  {cls_val:>2}  {name:<16}  {n:>8,}')

header          = laspy.LasHeader(point_format=las.header.point_format,
                                  version=las.header.version)
header.offsets  = las.header.offsets
header.scales   = las.header.scales
out             = laspy.LasData(header)
out.x, out.y, out.z = final_pts.T
out.classification   = final_cls
out.user_data        = final_pid
out.write(OUT_LAS)
print(f'{OUT_LAS} written')

# ── QML ──────────────────────────────────────────────────────────────────────
CLASS_COLORS = {
    65: (220, 80,  80,  'SHAFT'),
    66: (255, 200, 0,   'CROSSARM'),
    70: (60,  130, 220, 'SAME_LANE_WIRE'),
    71: (200, 50,  50,  'CROSS_ROAD_WIRE'),
    72: (120, 60,  200, 'TRANSFORMER'),
}
cats = ['    <category value="0" label="Unclassified" color="136,136,136,255" render="true"/>']
for cid, (r, g, b, name) in sorted(CLASS_COLORS.items()):
    cats.append(f'    <category value="{cid}" label="{name}" color="{r},{g},{b},255" render="true"/>')

qml = (
    "<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>\n"
    '<qgis styleCategories="Symbology" version="3.34">\n'
    '  <renderer-v2 type="pointcloudclassified" attribute="Classification">\n'
    '    <categories>\n'
    + '\n'.join(cats) + '\n'
    '    </categories>\n'
    '  </renderer-v2>\n'
    '</qgis>\n'
)
with open(OUT_QML, 'w') as f:
    f.write(qml)
print(f'{OUT_QML} written')
print('\n✅ Done. In QGIS: Layer Properties → Symbology → Load Style → poles_wires.qml')